In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

DATA_PATH = os.getcwd() + "/data"


/vol/bitbucket/gk825/nlpenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Torch: 2.2.2+cu118
CUDA available: True
GPU: NVIDIA GeForce RTX 4080


In [ ]:
pcl_df = pd.read_csv(
    DATA_PATH + '/dontpatronizeme_pcl.tsv', 
    delimiter='\t', 
    header=None)
pcl_df.columns = ['paragraph_id', 'article_id', 'keyword', 'country_code', 'paragraph', 'label']
pcl_df["y"] = pcl_df["label"].apply(lambda x: 0 if x < 2 else 1)
pcl_df['paragraph'] = pcl_df['paragraph'].fillna("").astype(str)

In [ ]:
train_ids = pd.read_csv( DATA_PATH + '/train_semeval_parids-labels.csv')
dev_ids = pd.read_csv(DATA_PATH + '/dev_semeval_parids-labels.csv')

train_df = pd.merge(pcl_df, train_ids, how='right', left_on='paragraph_id', right_on='par_id')
dev_df = pd.merge(pcl_df, dev_ids, how='right', left_on='paragraph_id', right_on='par_id')

# train_df["text"] = train_df["keyword"].astype(str) + " </s> " + train_df["paragraph"]
# dev_df["text"]  = dev_df["keyword"].astype(str)  + " </s> " + dev_df["paragraph"]
train_df["text"] = train_df["paragraph"]
dev_df["text"] = dev_df["paragraph"]


train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['y'])


In [ ]:
dev_df.head

<bound method NDFrame.head of       paragraph_id  article_id     keyword country_code  \
0             4046  @@14767805    hopeless           us   
1             1279   @@7896098     refugee           ng   
2             8330  @@17252299     refugee           ng   
3             4063   @@3002894     in-need           ie   
4             4089  @@25597822    homeless           pk   
...            ...         ...         ...          ...   
2089         10462  @@22092971    homeless           gh   
2090         10463   @@4676355     refugee           pk   
2091         10464  @@19612634    disabled           ie   
2092         10465  @@14297363       women           lk   
2093         10466  @@70091353  vulnerable           ph   

                                              paragraph  label_x  y  par_id  \
0     We also know that they can benefit by receivin...        3  1    4046   
1     Pope Francis washed and kissed the feet of Mus...        4  1    1279   
2     Many refugees do n

In [ ]:
train_dataset = Dataset.from_pandas(train_df[["text", "y"]])
val_dataset = Dataset.from_pandas(val_df[["text", "y"]])
dev_dataset = Dataset.from_pandas(dev_df[["text", "y"]])

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
dev_dataset = dev_dataset.rename_column("y", "labels")


train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dev_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])




/vol/bitbucket/gk825/nlpenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 2094/2094 [00:00<00:00, 3005.55 examples/s]


In [ ]:
neg_count = train_df[train_df.y == 0].y.count()
pos_count = train_df[train_df.y == 1].y.count()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_positive": f1}


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        total = neg_count + pos_count
        weight_neg = total / (2 * neg_count)
        weight_pos = total / (2 * pos_count)
        class_weights = torch.tensor([weight_neg, weight_pos], dtype=torch.float32)
        # class_weights = torch.tensor([1.0,neg_count / pos_count], dtype=torch.float32)
        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce_loss)

        focal_loss = (1 - pt) ** self.gamma * ce_loss

        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)  # 🔥 move to GPU
            at = alpha.gather(0, targets)
            focal_loss = at * focal_loss

        return focal_loss.mean()

class FocalTrainer(Trainer):
    def __init__(self, *args, alpha=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(alpha=alpha, gamma=gamma)

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss = self.focal_loss(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import RobertaModel

class CustomRobertaClassifier(nn.Module):
    def __init__(self, model_name="roberta-base"):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
        
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)
            
        return {"loss": loss, "logits": logits}

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    # "microsoft/deberta-v3-base",
    num_labels=2
)

# model = CustomRobertaClassifier()

# for name, param in model.roberta.encoder.layer[:6].named_parameters():
#     param.requires_grad = False

# from transformers import AutoModelForMaskedLM

# model = AutoModelForMaskedLM.from_pretrained(
#     "microsoft/deberta-v3-base"
# )

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,  # baseline = 1 epoch
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    learning_rate=1e-5,
    metric_for_best_model="f1_positive",
    greater_is_better=True
)

alpha = torch.tensor([0.25, 0.75])

# trainer = WeightedTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=val_dataset,
#     compute_metrics=compute_metrics,
# )

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    alpha=alpha,
    gamma=2.0
)

trainer.train()

metrics = trainer.evaluate()
print(metrics)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Positive
1,0.034500,0.027837,0.480808
2,0.023900,0.029460,0.559585
3,0.016000,0.041295,0.556430
4,0.009300,0.059644,0.574780
5,0.004500,0.073088,0.555891
6,0.003100,0.083122,0.569733


{'eval_loss': 0.05964371934533119, 'eval_f1_positive': 0.5747800586510264, 'eval_runtime': 4.1026, 'eval_samples_per_second': 408.28, 'eval_steps_per_second': 25.594, 'epoch': 6.0}


In [ ]:
preds_output = trainer.predict(dev_dataset)
logits = preds_output.predictions
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

print("Min prob:", probs.min())
print("Max prob:", probs.max())
print("Mean prob:", probs.mean())

Min prob: 0.016933843
Max prob: 0.95795226
Mean prob: 0.14920461


In [ ]:
preds_output = trainer.predict(val_dataset)
logits = preds_output.predictions
labels = preds_output.label_ids

probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()

best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.1, 0.9, 0.01):
    preds_t = (probs > t).astype(int)
    f1 = f1_score(labels, preds_t, pos_label=1)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best threshold:", best_thresh)
print("Best F1:", best_f1)

Best threshold: 0.46999999999999986
Best F1: 0.5902578796561605


In [ ]:
preds_output = trainer.predict(dev_dataset)
logits = preds_output.predictions
labels = preds_output.label_ids
probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
preds_t = (probs > 0.35).astype(int)
f1 = f1_score(labels, preds_t, pos_label=1)
print(f1)

0.5875


In [ ]:
with open("dev.txt", "w") as f:
    for item in preds_t:
        f.write(str(item) + "\n")

In [ ]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def apply_prompt(text):
    return f"{text} This paragraph is {tokenizer.mask_token}."

train_df["text"] = train_df["paragraph"].apply(apply_prompt)
val_df["text"] = val_df["paragraph"].apply(apply_prompt)
dev_df["text"] = dev_df["paragraph"].apply(apply_prompt)

train_dataset = Dataset.from_pandas(train_df[["text", "y"]])
val_dataset = Dataset.from_pandas(val_df[["text", "y"]])
dev_dataset = Dataset.from_pandas(dev_df[["text", "y"]])

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
dev_dataset = dev_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("y", "labels")
val_dataset = val_dataset.rename_column("y", "labels")
dev_dataset = dev_dataset.rename_column("y", "labels")

train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)
val_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)
dev_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

positive_word = "patronizing"
negative_word = "neutral"

print("Tokenized positive word:", tokenizer.tokenize(positive_word))
print("Tokenized negative word:", tokenizer.tokenize(negative_word))

pos_token_id = tokenizer.convert_tokens_to_ids(positive_word)
neg_token_id = tokenizer.convert_tokens_to_ids(negative_word)

model = AutoModelForMaskedLM.from_pretrained(model_name)

class PromptTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits  # (batch, seq_len, vocab)

        # find mask positions
        mask_positions = (inputs["input_ids"] == tokenizer.mask_token_id)

        # get logits at mask
        mask_logits = logits[mask_positions]

        # build 2-class logits from vocab
        selected_logits = torch.stack([
            mask_logits[:, neg_token_id],
            mask_logits[:, pos_token_id]
        ], dim=1)

        loss = F.cross_entropy(selected_logits, labels)

        return (loss, selected_logits) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds)
    return {"f1_positive": f1}

training_args = TrainingArguments(
    output_dir="./results_prompt",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_positive",
    greater_is_better=True,
    learning_rate=2e-5,
    report_to="none"
)

trainer = PromptTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()
print(metrics)

NameError: name 'AutoTokenizer' is not defined